# Chatbot with Similarity Score

This notebook uses the fundamental code of the rubin_rag chatbot to perform similarity searches and return similarity scores along with the RAG LLM response. It is missing the chat history functionality of the full chatbot, but the user can receive a single answer to a single query. This is useful for viewing the quality of context retrieved by the RAG and the relation of the answer provided by the LLM to that retrieved context.

In [ ]:
import os
import warnings
from dotenv import load_dotenv
from collections.abc import Callable
from typing import Any
import logging

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.vectorstores.base import VectorStoreRetriever
from langchain_core.runnables import Runnable
from langchain_core.runnables import RunnableLambda
from langchain_core.documents.base import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_weaviate.vectorstores import WeaviateVectorStore

import weaviate
from weaviate.classes.query import MetadataQuery, Filter
from weaviate.classes.init import Auth
from weaviate.classes.query import Filter
from weaviate.client import WeaviateClient

In [ ]:
# Suppress all warnings
warnings.filterwarnings("ignore")

# Load environment variables
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

In [ ]:
class CustomWeaviateVectorStore(WeaviateVectorStore):
    """Custom Vector Store overrides the similarity search function."""

    def __init__(
        self,
        client: Any,
        index_name: str,
        text_key: str,
        embedding: Any,
        attributes: list | None = None,
        relevance_score_fn: Callable | None = None,
        use_multi_tenancy: bool | None = None,
    ) -> None:
        """Initialize the CustomWeaviateVectorStore class."""
        if use_multi_tenancy is None:
            use_multi_tenancy = False

        self.client = client
        self.index_name = index_name
        self.text_key = text_key
        self.embedding = embedding

        super().__init__(
            client=client,
            index_name=index_name,
            text_key=text_key,
            embedding=embedding,
            attributes=attributes,
            relevance_score_fn=relevance_score_fn,
            use_multi_tenancy=use_multi_tenancy,
        )

    def similarity_search(
        self, query: str, k: int = 4, **kwargs: Any
    ) -> list[Document]:
        """
        Return list of documents most similar to the query text and their
        score. A higher score means more similarity, with a max of 1.
        """
        where_filter = kwargs.get("where_filter")
        collection = self.client.collections.get(self.index_name)
        response = collection.query.hybrid(
            query=query,
            limit=k,
            filters=where_filter,
            alpha=1,
            return_metadata=MetadataQuery(score=True, explain_score=True),
        )

        results = []
        for obj in response.objects:
            text = obj.properties.get("page_content", "")
            metadata = obj.properties.copy() if obj.properties else {}
            metadata["score"] = (
                obj.metadata.score
            )  # Inject the score into metadata
            results.append(Document(page_content=text, metadata=metadata))
        return results

In [ ]:
def configure_client() -> WeaviateClient:
    """Configure the Weaviate client."""
    openai_api_key = os.getenv("OPENAI_API_KEY")
    weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
    http_host = os.getenv("HTTP_HOST")
    grpc_host = os.getenv("GRPC_HOST")

    if openai_api_key is None:
        raise ValueError("OPENAI_API_KEY environment variable is not set")
    if weaviate_api_key is None:
        raise ValueError("WEAVIATE_API_KEY environment variable is not set")
    if http_host is None:
        raise ValueError("HTTP_HOST environment variable is not set")
    if grpc_host is None:
        raise ValueError("GRPC_HOST environment variable is not set")

    return weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,  # Database on port 80 in USDF
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,
        grpc_secure=False,
        auth_credentials=Auth.api_key(weaviate_api_key),
        headers={"X-OpenAI-Api-Key": openai_api_key},
        skip_init_checks=True,
    )


def configure_retriever() -> VectorStoreRetriever:
    """Configure the Weaviate retriever."""
    selected_sources = [
        "github", "jira", "lsst_bib", "webpage", "discourse"
    ]
    if selected_sources:
        filters = Filter.by_property("source_key").contains_any(
            selected_sources
        )

    search_kwargs = {
        "k": 6,
        "where_filter": filters
    }

    return CustomWeaviateVectorStore(
        client=configure_client(),
        index_name="Ingestion_20250610",
        text_key="page_content",
        embedding=OpenAIEmbeddings(
            model="text-embedding-3-small", dimensions=1536
        ),
        attributes=["source", "source_key"],  # Metadata to fetch
    ).as_retriever(
        search_type="similarity",
        search_kwargs=search_kwargs,
    )


def create_qa_chain(
    input_retriever: Callable[[str], list[Document]],
) -> Runnable:
    """Create a QA chain for the chatbot using a custom retriever."""

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, streaming=True)

    system_template = """You are Rubin AI Assistant, a helpful assistant at
    Vera C Rubin Observatory.
    Do your best to answer the questions in as much detail as possible.
    Do not attempt to provide an answer if you do not know the answer.
    In your response, do not recommend reading elsewhere.
    Use the following pieces of context to answer the user's
    question at the end.
    ----------------
    {context}
    ----------------"""

    qa_prompt = ChatPromptTemplate.from_messages(
        [
            SystemMessagePromptTemplate.from_template(system_template),
            MessagesPlaceholder("chat_history"),
            HumanMessagePromptTemplate.from_template("Question:```{input}```"),
        ]
    )

    question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
    
    retriever_runnable = RunnableLambda(input_retriever)
    return create_retrieval_chain(retriever_runnable, question_answer_chain)

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host="weaviate-headless.rubin-rag.svc.cluster.local",
        http_port=8080,
        http_secure=False,
        grpc_host="weaviate-grpc.rubin-rag.svc.cluster.local",
        grpc_port=50051,
        grpc_secure=False,
        auth_credentials=Auth.api_key(os.getenv("WEAVIATE_API_KEY")),
        headers={"X-OpenAI-Api-Key": os.getenv("OPENAI_API_KEY")},
        skip_init_checks=True,
    )
    
    user_query = "wWhat is LSST Cam?"
    retriever = configure_retriever()
    qa_chain = create_qa_chain(retriever)

    result = qa_chain.invoke(
        {
            "input": user_query,
            "chat_history": []
        }
    )
    print("Input:\n", result["input"])
    print("\nAnswer:\n", result["answer"])

    print("\nContext Documents:")
    for i, doc in enumerate(result["context"], 1):
        print(f"\n--- Document {i} ---")
        print("Similarity Score:", doc.metadata.get("score"))
        print("Source:", doc.metadata.get("source"))
        print("Content:\n", doc.page_content)


except Exception as e:
    print(f"An error occurred: {e}")
finally:
    client.close()